In [3]:
# ===========================================
# Notebook 05 — Feature Engineering (Fixed)
# ===========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use("seaborn-v0_8")

# Load Data

# Clean OHLCV + log returns (from Notebook 01)
df_raw = pd.read_csv(
    "../data/raw/VIX_daily.csv",
    parse_dates=["Date"]
).set_index("Date")

# Realized volatility (from Notebook 02)
df_rv = pd.read_csv(
    "../data/processed/realized_vol.csv",
    parse_dates=["Date"]
).set_index("Date")

# Construct Base Feature Frame (FIXED)
df = df_raw[["Open", "High", "Low", "Close", "Volume"]].copy()
df = df.join(df_rv[["RV_daily_sq", "RV_21", "Parkinson_21"]], how="inner")

# Compute clean log returns
df["log_ret"] = np.log(df["Close"]).diff()

# Define target RV
df["RV_target"] = df["RV_21"]

## Lagged Features

# Return lags
for lag in [1, 2, 5, 10, 21]:
    df[f"log_ret_lag{lag}"] = df["log_ret"].shift(lag)

# RV lags
for lag in [1, 2, 5, 10, 21]:
    df[f"RV_lag{lag}"] = df["RV_target"].shift(lag)

# Rolling Window Features
def add_rolling_features(series, name):
    for win in [5, 10, 21, 42]:
        df[f"{name}_roll{win}_mean"] = series.rolling(win).mean()
        df[f"{name}_roll{win}_std"]  = series.rolling(win).std()
        df[f"{name}_roll{win}_min"]  = series.rolling(win).min()
        df[f"{name}_roll{win}_max"]  = series.rolling(win).max()

# Use absolute returns & RV for rolling stats
add_rolling_features(df["log_ret"].abs(), "absret")
add_rolling_features(df["RV_target"], "RV")

# Range-Based Volatility Features
df["parkinson"] = (np.log(df["High"] / df["Low"]) ** 2) / (4 * np.log(2))

df["garman_klass"] = (
    0.5 * (np.log(df["High"] / df["Low"]) ** 2)
    - (2 * np.log(2) - 1) * (np.log(df["Close"] / df["Open"]) ** 2)
)

## Volatility Regime Indicators

# High RV regime (top 20%)
threshold = df["RV_target"].quantile(0.80)
df["regime_high_vol"] = (df["RV_target"] > threshold).astype(int)

# Volatility spike: 50% above 21-day mean RV
df["vol_spike"] = (
    df["RV_target"] > df["RV_target"].rolling(21).mean() * 1.5
).astype(int)

# Return spike: 200% above 21-day mean abs returns
df["ret_spike"] = (
    df["log_ret"].abs() > df["log_ret"].abs().rolling(21).mean() * 2
).astype(int)

# VIX-Specific Macro Features
df["VIX_level"] = df["Close"]
df["VIX_change"] = df["Close"].pct_change()
df["vol_of_vol"] = df["RV_target"].pct_change()

# Add GARCH & HAR Forecasts (If Available)
garch_path = "../data/processed/garch_forecasts.csv"
har_path   = "../data/processed/har_forecasts.csv"

if os.path.exists(garch_path):
    df_garch = pd.read_csv(garch_path, parse_dates=["Date"]).set_index("Date")
    df = df.join(df_garch[["GARCH_vol_forecast"]], how="left")

if os.path.exists(har_path):
    df_har = pd.read_csv(har_path, parse_dates=["Date"]).set_index("Date")
    df = df.join(df_har[["HAR_RV_forecast"]], how="left")

# Drop NaNs and finalize feature matrix
df_final = df.dropna()

print("\nFinalized feature dataset:")
display(df_final.head())

# Save Final Set
os.makedirs("../data/processed", exist_ok=True)
out_path = "../data/processed/features.csv"
df_final.to_csv(out_path)
print("\nSaved features to:", out_path)


Finalized feature dataset:


,Open,High,Low,Close,Volume,RV_daily_sq,RV_21,Parkinson_21,log_ret,RV_target,...,parkinson,garman_klass,regime_high_vol,vol_spike,ret_spike,VIX_level,VIX_change,vol_of_vol,GARCH_vol_forecast,HAR_RV_forecast
Date,,,,,,,,,,,,,,,,,,,,,
2023-04-26,18.660000,19.610001,17.870001,18.840000,0,0.000018,0.163710,0.200321,0.004255,0.163710,...,0.003114,0.004281,0,0,0,18.840000,0.004264,-0.049803,0.075682,0.000918
2023-04-27,18.430000,18.430000,16.719999,17.030001,0,0.010202,0.189838,0.204132,-0.101006,0.189838,...,0.003420,0.002330,0,0,1,17.030001,-0.096072,0.159598,0.068660,0.000768
2023-04-28,17.209999,17.650000,15.720000,15.780000,0,0.005812,0.199895,0.215362,-0.076233,0.199895,...,0.004837,0.003798,0,0,1,15.780000,-0.073400,0.052977,0.077984,0.001116
2023-05-01,16.410000,16.620001,15.530000,16.080000,0,0.000355,0.200712,0.215868,0.018833,0.200712,...,0.001660,0.002141,0,0,0,16.080000,0.019011,0.004086,0.077764,0.001217
2023-05-02,16.270000,19.809999,16.260000,17.780001,0,0.010100,0.223824,0.244611,0.100498,0.223824,...,0.014065,0.016456,0,0,1,17.780001,0.105721,0.115151,0.070892,0.001135



Saved features to: ../data/processed/features.csv
